In [2]:
import pandas as pd
from dotenv import load_dotenv
import os
from sqlalchemy import create_engine, text
from datetime import datetime
from dateutil.relativedelta import relativedelta

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [3]:
class Updater:
    def __init__(self):
        
        self.__get_engine()
        
    def __load_dotenv(self):
        load_dotenv()
        return {
            'user': 'admin',
            'password': '1234',
            'host': 'localhost',
            'port': 5432,
            'db_name': 'skud_va',
            'schema': 'fin'
        }

    def __get_engine(self):
        conf = self.__load_dotenv()
        self.__engine = create_engine(f"postgresql+psycopg2://{conf['user']}:{conf['password']}@{conf['host']}:{conf['port']}/{conf['db_name']}")

    def read_sql_query(self, query):
        with self.__engine.begin() as con:
            return pd.read_sql_query(query, con)

    def execute_sql(self, query):
        with self.__engine.connect() as con:
            con.execute(text(query))
            con.commit()

    def pandas_to_db(self, df, table, exists):
        with self.__engine.connect() as con:
            df.to_sql(table, con, if_exists=exists, index=False, schema='fin')

worker = Updater()

filename = 'forecast_form.xlsx'

Таблицы в excel:
- Справочник ЦФО и ответственных </br>
- Матрица  -> Какие статьи у какого цфо </br>    --> `costs_frc_mapping`
- mapping   -> Какие статьи к каким статьям для консолидации принадлежат  --> `costs_consolidation_mapping`

In [30]:
frc_cost_users = pd.read_excel(filename, sheet_name='Справочник ЦФО и ответственных')
frc_center = pd.read_excel(filename, sheet_name='Матрица')
consolidation_costs = pd.read_excel(filename, sheet_name='mapping')

In [15]:
frc_center = ( 
    frc_center
    .rename(columns={"Статья 1С": "type_1c", "ЦФО-планировщик": "frc"})
    [['type_1c', 'frc']]
)

In [17]:
worker.pandas_to_db(frc_center, 'cost_frc_mapping', 'append')
# Create constraint to unique both columns

In [21]:
cc = (
    consolidation_costs
    .rename(columns={
        "Статья 1С": "type_1c",
        "Статья для консолидации": "cons_type",
    })
)

In [22]:
worker.pandas_to_db(cc, 'cost_consolidate_mapping', 'append')
# The same, create constraint unique both columns

In [31]:
frc_cost_users['user'] = frc_cost_users['Ф'] + ' ' + frc_cost_users['И'] + ' ' + frc_cost_users['О']
frc_cost_users = (
    frc_cost_users
    .assign(
        login = lambda x:  x['почта'].apply(lambda x: x.split('@')[0]),
    )
    .rename(columns={'Наименование ЦФО': 'frc', 'почта': 'email'})
    [['frc', 'user', 'email', 'login']]
)

In [56]:
s = """insert into fin.frc_user (frc, "user", email, login, is_cost) values """
for i, item in frc_cost_users.iterrows():
    s += f"""('{item.frc}', '{item.user}', '{item.email}', '{item.login}', 1), """
s = s[:-2] + " on conflict (frc, login) do update set is_cost = 1;"


In [57]:
worker.execute_sql(s)

### Updating cost_est

CREATE TABLE fin.cost_est (
	id int8 GENERATED ALWAYS AS IDENTITY( INCREMENT BY 1 MINVALUE 1 MAXVALUE 9223372036854775807 START 1 CACHE 1 NO CYCLE) NOT NULL,
	company text NULL,
	date_dt date NULL,
	estimate_date date NULL,
	frc varchar(100) NULL,
	cons_type varchar(100) NULL,
	type_1c varchar(100) NULL,
	frc_owner varchar(100) NULL,
	amount float4 NULL,
	CONSTRAINT cost_est_pkey PRIMARY KEY (id)
);

In [42]:
query = """
    select max(ce.estimate_date) as max_dt
    from fin.cost_est ce
"""
max_est_dt = worker.read_sql_query(query)['max_dt'].values[0]

In [45]:
estimate_date = datetime.now().date().replace(day=1)
current_month = datetime.now().month

# if max_est_dt == estimate_date:
#     print("The estimate date already exists")

In [9]:
dates = [estimate_date.replace(month=i).strftime("%Y-%m-%d") for i in range(current_month, 13)]

In [24]:
dates_df = pd.DataFrame({"date_dt": dates})

In [68]:
query = """
    select fm.frc as frc_owner, 
        fm.type_1c, 
        cm.cons_type
    from fin.cost_frc_mapping fm
    left join fin.cost_consolidate_mapping cm
    on fm.type_1c = cm.type_1c
    """
df = worker.read_sql_query(query)

In [69]:
cost_est_append = (
    df
    .assign(
        estimate_date = estimate_date,
        company =  'АО "РТ-Техприемка"',
    )
    .merge(dates_df, how="cross")
    [['company', 'date_dt', 'estimate_date', 'cons_type', 'type_1c', 'frc_owner']]
    .sort_values(['frc_owner', 'cons_type', 'type_1c', 'date_dt'])
)

In [70]:
s = """insert into fin.cost_est (company, date_dt, estimate_date, cons_type, type_1c, frc_owner) 
        values """

for _, item in cost_est_append.iterrows():
    s += f"""('{item.company}', '{item.date_dt}', 
        '{item.estimate_date}', '{item.cons_type}', 
        '{item.type_1c}', '{item.frc_owner}'), """
s = s[:-2] + """ on conflict 
        (company, date_dt, estimate_date, cons_type, type_1c, frc_owner) 
        do nothing;"""

In [72]:
worker.execute_sql(s)